In [28]:
import json, re
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd
import requests
from time import sleep


# the below functions will examine the weirdly-encoded JSON objects in the Met HTML data

# finds all encoded JSON data
def rsc_payload(html):
    """Join all self.__next_f.push([1,"..."]) chunks into one decoded string."""
    soup = BeautifulSoup(html, "html.parser")
    chunks = []
    for script in soup.find_all("script"):
        text = script.string or ""
        for m in re.finditer(r'self\.__next_f\.push\(\[1,\s*(".*?")\]\)', text, re.S):
            chunks.append(json.loads(m.group(1)))   # un-escapes \" and \u003c etc.
    return "".join(chunks)

# assemble the encoded data into readable sections
def find_html(node):
    """Recursively collect every dangerouslySetInnerHTML.__html string."""
    if isinstance(node, dict):
        if "dangerouslySetInnerHTML" in node:
            yield node["dangerouslySetInnerHTML"]["__html"]
        for v in node.values():
            yield from find_html(v)
    elif isinstance(node, list):
        for v in node:
            yield from find_html(v)

# will find a part of that JSON object, such as provenance
def get_section(html, name="Provenance"):
    payload = rsc_payload(html)
    start = payload.find('{"name":' + json.dumps(name))
    if start == -1:
        return None
    obj, _ = json.JSONDecoder().raw_decode(payload, start)  # parses exactly one object
    raw = "".join(find_html(obj["body"]))
    return BeautifulSoup(raw.replace("<br/>", "\n"), "html.parser").get_text().strip()

# will get the Object ID, which we need later to query API based on our scraped data
def get_object_id(html):
    """Return the Met object ID (same as the Met API's objectID), or None."""
    soup = BeautifulSoup(html, "html.parser")
    # Met pages include <script type="application/json" id="analytics">{"objectID":311021,...}</script>
    tag = soup.find("script", id="analytics", type="application/json")
    if tag and tag.string:
        oid = json.loads(tag.string).get("objectID")
        if oid is not None:
            return int(oid)
    # Fallback: search the hidden Next.js data for "objectID":"311021"
    m = re.search(r'"objectID":"?(\d+)', rsc_payload(html))
    return int(m.group(1)) if m else None



In [11]:
met_objects = []         # first we make an empty list to contain our met objects scraped from pages
for path in sorted(Path("met_pages").glob("*.html")): # go through every single HTML file in the folder
    html = path.read_text(encoding="utf-8")  # read the html file as text
    met_objects.append(   # builds a dictionary using the scraped info from each object:
    {
        "page_name": path.stem,  #page name
        "object_id": get_object_id(html),  # uses our get_object_id function to extract object ID
        "provenance": get_section(html, "Provenance"), # uses get_section function to extract provenance
        "html": html, # stores HTML data 
    })

len(met_objects)  # to confirm that it has scraped everything we check the length of our met_objects list

210

In [25]:
df = pd.DataFrame(met_objects)
df = df.drop(columns="html")
df.tail()

,page_name,object_id,provenance
205,Yoruba bead artist - Adéńlá (big crown) - Y...,310177,"[Ernst Anspach, New York, by 1976]"
206,Zlan of Belewale - Female portrait lü me (fig...,310960,"[Henri Kamer, Paris and New York, by 1964]; Th..."
207,Zlan of Belewale - Wunkirmian or wakemia feast...,312230,"Collection Maurice Girardin, Paris, before 195..."
208,_Barnes Foundation Master_ - Seated couple - D...,310325,"[Henri Kamer, Paris and New York]; Lester Wund..."
209,_Master of Ogol_ - Female dege dal nda (sculpt...,310330,"Lester Wunderman, New York, until 1977"


In [34]:
# now we use another API endpoint for object data rather than search results
object_to_find = 310325

base_url = "https://collectionapi.metmuseum.org"
object_path = '/public/collection/v1/objects/'
url = f"{base_url}{object_path}{object_to_find}"

response_object = requests.get(url)
parsed_response = response_object.json()

In [35]:
parsed_response.keys()

dict_keys(['objectID', 'isHighlight', 'accessionNumber', 'accessionYear', 'isPublicDomain', 'primaryImage', 'primaryImageSmall', 'additionalImages', 'constituents', 'department', 'objectName', 'title', 'culture', 'period', 'dynasty', 'reign', 'portfolio', 'artistRole', 'artistPrefix', 'artistDisplayName', 'artistDisplayBio', 'artistSuffix', 'artistAlphaSort', 'artistNationality', 'artistBeginDate', 'artistEndDate', 'artistGender', 'artistWikidata_URL', 'artistULAN_URL', 'objectDate', 'objectBeginDate', 'objectEndDate', 'medium', 'dimensions', 'measurements', 'creditLine', 'geographyType', 'city', 'state', 'county', 'country', 'region', 'subregion', 'locale', 'locus', 'excavation', 'river', 'classification', 'rightsAndReproduction', 'linkResource', 'metadataDate', 'repository', 'objectURL', 'tags', 'objectWikidata_URL', 'isTimelineWork', 'GalleryNumber'])

In [48]:
met_api_data = []  # we will store all the API data we have grabbed

for met_object in met_objects:
    obj_id = met_object["object_id"] #finding the object_id in the list
    
    url = f"{base_url}{object_path}{obj_id}" # building an API req URL

    response_object = requests.get(url) # grabbing the data from the API
    parsed_response = response_object.json() #parsing it as JSON
    met_api_data.append(parsed_response)
    print("grabbed id ", obj_id)
    sleep(1.5)  # wait 1.5 seconds to not overrun API limits

len(met_api_data)

grabbed id  310842
grabbed id  312390
grabbed id  310871
grabbed id  310913
grabbed id  310829
grabbed id  312226
grabbed id  312427
grabbed id  312428
grabbed id  310826
grabbed id  310825
grabbed id  312338
grabbed id  312189
grabbed id  312437
grabbed id  312270
grabbed id  312304
grabbed id  310781
grabbed id  310744
grabbed id  312285
grabbed id  312389
grabbed id  310780
grabbed id  312257
grabbed id  312358
grabbed id  312379
grabbed id  310789
grabbed id  310916
grabbed id  312439
grabbed id  312373
grabbed id  312376
grabbed id  312184
grabbed id  312370
grabbed id  312365
grabbed id  310800
grabbed id  309843
grabbed id  312335
grabbed id  312316
grabbed id  312303
grabbed id  310732
grabbed id  310307
grabbed id  310930
grabbed id  319458
grabbed id  310854
grabbed id  312154
grabbed id  312172
grabbed id  310733
grabbed id  312442
grabbed id  311004
grabbed id  312196
grabbed id  312429
grabbed id  312372
grabbed id  309997
grabbed id  309909
grabbed id  310853
grabbed id  

210

In [49]:
# let's save the met_api_json and scraped_provenance_json as files

with open("met_api_object_data.json", "w") as file:
    json.dump(met_api_data, file, indent=4)

with open("met_scraped_provenance_data.json", "w") as file:
    json.dump(met_objects, file, indent=4)